In [1]:
import io
import numpy as np
import pandas as pd
import pyarrow as pa

# **CSV – basic reading/writing**

> generally, **CSV** rates as one of the worst formats for **CPU** efficiency, memory usage, and losslessness.

In [3]:
df = pd.DataFrame([
["Paul", "McCartney", 1942],
["John", "Lennon", 1940],
["Richard", "Starkey", 1940],
["George", "Harrison", 1943],
], columns=["first", "last", "birth"])

df

,first,last,birth
0,Paul,McCartney,1942
1,John,Lennon,1940
2,Richard,Starkey,1940
3,George,Harrison,1943


In [4]:
df.dtypes

first    object
last     object
birth     int64
dtype: object

In [5]:
df = df.convert_dtypes(dtype_backend="numpy_nullable")

df.dtypes

first    string[python]
last     string[python]
birth             Int64
dtype: object

In [6]:
import io

In [15]:
buf = io.StringIO()
df.to_csv(buf)

print(buf.getvalue())

,first,last,birth
0,Paul,McCartney,1942
1,John,Lennon,1940
2,Richard,Starkey,1940
3,George,Harrison,1943



In [16]:
buf.seek(0)
pd.read_csv(buf, dtype_backend="numpy_nullable")

,Unnamed: 0,first,last,birth
0,0,Paul,McCartney,1942
1,1,John,Lennon,1940
2,2,Richard,Starkey,1940
3,3,George,Harrison,1943


In [17]:
buf.seek(0)
pd.read_csv(buf, dtype_backend="numpy_nullable", index_col=0)

,first,last,birth
0,Paul,McCartney,1942
1,John,Lennon,1940
2,Richard,Starkey,1940
3,George,Harrison,1943


In [18]:
buf.seek(0)
pd.read_csv(buf, dtype_backend="numpy_nullable", index_col="Unnamed: 0")

,first,last,birth
0,Paul,McCartney,1942
1,John,Lennon,1940
2,Richard,Starkey,1940
3,George,Harrison,1943


In [20]:
buf = io.StringIO()
df.to_csv(buf, index=False)
print(buf.getvalue())

first,last,birth
Paul,McCartney,1942
John,Lennon,1940
Richard,Starkey,1940
George,Harrison,1943



In [22]:
df = pd.DataFrame([
["McCartney, Paul", 1942],
["Lennon, John", 1940],
["Starkey, Richard", 1940],
["Harrison, George", 1943],
], columns=["name", "birth"]).convert_dtypes(dtype_backend="numpy_nullable")

df

,name,birth
0,"McCartney, Paul",1942
1,"Lennon, John",1940
2,"Starkey, Richard",1940
3,"Harrison, George",1943


In [23]:
df.dtypes

name     string[python]
birth             Int64
dtype: object

In [24]:
buf = io.StringIO()
df.to_csv(buf, index=False)

print(buf.getvalue())

name,birth
"McCartney, Paul",1942
"Lennon, John",1940
"Starkey, Richard",1940
"Harrison, George",1943



In [25]:
buf = io.StringIO()
df.to_csv(buf, index=False, sep="|")

print(buf.getvalue())

name|birth
McCartney, Paul|1942
Lennon, John|1940
Starkey, Richard|1940
Harrison, George|1943



In [26]:
df = pd.DataFrame({
    "col1": ["a"] * 1_000,
    "col2": ["b"] * 1_000,
    "col3": ["c"] * 1_000,
}).convert_dtypes(dtype_backend="numpy_nullable")

df.head()

,col1,col2,col3
0,a,b,c
1,a,b,c
2,a,b,c
3,a,b,c
4,a,b,c


In [29]:
buf = io.StringIO()
df.to_csv(buf, index=False)

len(buf.getvalue())

6015

In [30]:
buf = io.BytesIO()
df.to_csv(buf, index=False, compression="gzip")

len(buf.getvalue())

69

> The trade-off here is that while compressed files require less disk storage, they require more work from the CPU to compress or decompress the file contents.

# **CSV – strategies for reading large files**

In [2]:
df = pd.read_csv('../Datasets/diamonds.csv', dtype_backend="numpy_nullable", nrows=1_000)

df

,carat,cut,color,clarity,depth,table,price,x,y,z
0,0.23,Ideal,E,SI2,61.5,55.0,326,3.95,3.98,2.43
1,0.21,Premium,E,SI1,59.8,61.0,326,3.89,3.84,2.31
2,0.23,Good,E,VS1,56.9,65.0,327,4.05,4.07,2.31
3,0.29,Premium,I,VS2,62.4,58.0,334,4.2,4.23,2.63
4,0.31,Good,J,SI2,63.3,58.0,335,4.34,4.35,2.75
...,...,...,...,...,...,...,...,...,...,...
995,0.54,Ideal,D,VVS2,61.4,52.0,2897,5.3,5.34,3.26
996,0.72,Ideal,E,SI1,62.5,55.0,2897,5.69,5.74,3.57
997,0.72,Good,F,VS1,59.4,61.0,2897,5.82,5.89,3.48
998,0.74,Premium,D,VS2,61.8,58.0,2897,5.81,5.77,3.58


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 10 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   carat    1000 non-null   Float64
 1   cut      1000 non-null   string 
 2   color    1000 non-null   string 
 3   clarity  1000 non-null   string 
 4   depth    1000 non-null   Float64
 5   table    1000 non-null   Float64
 6   price    1000 non-null   Int64  
 7   x        1000 non-null   Float64
 8   y        1000 non-null   Float64
 9   z        1000 non-null   Float64
dtypes: Float64(6), Int64(1), string(3)
memory usage: 85.1 KB


In [6]:
df[["price"]].describe()

,price
count,1000.0
mean,2476.54
std,839.57562
min,326.0
25%,2777.0
50%,2818.0
75%,2856.0
max,2898.0


In [7]:
df[["carat"]].describe()

,carat
count,1000.0
mean,0.68928
std,0.195291
min,0.2
25%,0.7
50%,0.71
75%,0.79
max,1.27


In [9]:
df2 = pd.read_csv(
    "../Datasets/diamonds.csv",
    nrows=1_000,
    dtype={
        "carat": pd.Float32Dtype(),
        "cut": pd.StringDtype(),
        "color": pd.StringDtype(),
        "clarity": pd.StringDtype(),
        "depth": pd.Float32Dtype(),
        "table": pd.Float32Dtype(),
        "price": pd.Int16Dtype(),
        "x": pd.Float32Dtype(),
        "y": pd.Float32Dtype(),
        "z": pd.Float32Dtype(),
    }
)

df2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 10 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   carat    1000 non-null   Float32
 1   cut      1000 non-null   string 
 2   color    1000 non-null   string 
 3   clarity  1000 non-null   string 
 4   depth    1000 non-null   Float32
 5   table    1000 non-null   Float32
 6   price    1000 non-null   Int16  
 7   x        1000 non-null   Float32
 8   y        1000 non-null   Float32
 9   z        1000 non-null   Float32
dtypes: Float32(6), Int16(1), string(3)
memory usage: 55.8 KB


In [11]:
df.describe()

,carat,depth,table,price,x,y,z
count,1000.0,1000.0,1000.0,1000.0,1000.0,1000.0,1000.0
mean,0.68928,61.7228,57.7347,2476.54,5.60594,5.59918,3.45753
std,0.195291,1.758879,2.467946,839.57562,0.625173,0.611974,0.389819
min,0.2,53.0,52.0,326.0,3.79,3.75,2.27
25%,0.7,60.9,56.0,2777.0,5.64,5.63,3.45
50%,0.71,61.8,57.0,2818.0,5.77,5.76,3.55
75%,0.79,62.6,59.0,2856.0,5.92,5.91,3.64
max,1.27,69.5,70.0,2898.0,7.12,7.05,4.33


In [12]:
df2.describe()

,carat,depth,table,price,x,y,z
count,1000.0,1000.0,1000.0,1000.0,1000.0,1000.0,1000.0
mean,0.68928,61.722801,57.734699,2476.54,5.60594,5.59918,3.45753
std,0.195291,1.758879,2.467946,839.57562,0.625173,0.611974,0.389819
min,0.2,53.0,52.0,326.0,3.79,3.75,2.27
25%,0.7,60.900002,56.0,2777.0,5.64,5.63,3.45
50%,0.71,61.799999,57.0,2818.0,5.77,5.76,3.55
75%,0.79,62.599998,59.0,2856.0,5.92,5.91,3.64
max,1.27,69.5,70.0,2898.0,7.12,7.05,4.33


In [14]:
df["cut"].unique()

<StringArray>
['Ideal', 'Premium', 'Good', 'Very Good', 'Fair']
Length: 5, dtype: string

In [15]:
df["color"].unique()

<StringArray>
['E', 'I', 'J', 'H', 'F', 'G', 'D']
Length: 7, dtype: string

In [16]:
df["clarity"].unique()

<StringArray>
['SI2', 'SI1', 'VS1', 'VS2', 'VVS2', 'VVS1', 'I1', 'IF']
Length: 8, dtype: string

> **Note :** However, I would advise against using **pd.CategoricalDtype()** as an argument to **dtype=** , as by default it uses **np.nan** as a missing value indicator (for a refresher on this caveat, you may want to revisit the **Categorical types** recipe back in Chapter 3, Data Types). Instead, the best approach to convert your strings to categorical types is to first read in your columns as **pd.StringDtype()** , and then use **pd.DataFrame.astype** on the appropriate column(s)

In [17]:
df3 = pd.read_csv(
    "../Datasets/diamonds.csv",
    nrows=1_000,
    dtype={
        "carat": pd.Float32Dtype(),
        "cut": pd.StringDtype(),
        "color": pd.StringDtype(),
        "clarity": pd.StringDtype(),
        "depth": pd.Float32Dtype(),
        "table": pd.Float32Dtype(),
        "price": pd.Int16Dtype(),
        "x": pd.Float32Dtype(),
        "y": pd.Float32Dtype(),
        "z": pd.Float32Dtype(),
    }
)

cat_cols = ["cut", "color", "clarity"]
df3[cat_cols] = df3[cat_cols].astype(pd.CategoricalDtype())

df3.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 10 columns):
 #   Column   Non-Null Count  Dtype   
---  ------   --------------  -----   
 0   carat    1000 non-null   Float32 
 1   cut      1000 non-null   category
 2   color    1000 non-null   category
 3   clarity  1000 non-null   category
 4   depth    1000 non-null   Float32 
 5   table    1000 non-null   Float32 
 6   price    1000 non-null   Int16   
 7   x        1000 non-null   Float32 
 8   y        1000 non-null   Float32 
 9   z        1000 non-null   Float32 
dtypes: Float32(6), Int16(1), category(3)
memory usage: 36.2 KB


In [18]:
dtypes = { # does not include x, y, or z
        "carat": pd.Float32Dtype(),
        "cut": pd.StringDtype(),
        "color": pd.StringDtype(),
        "clarity": pd.StringDtype(),
        "depth": pd.Float32Dtype(),
        "table": pd.Float32Dtype(),
        "price": pd.Int16Dtype(),
}

df4 = pd.read_csv(
    "../Datasets/diamonds.csv",
    nrows=1_000,
    dtype=dtypes,
    usecols=dtypes.keys(),
)

cat_cols = ["cut", "color", "clarity"]
df4[cat_cols] = df4[cat_cols].astype(pd.CategoricalDtype())

df4.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 7 columns):
 #   Column   Non-Null Count  Dtype   
---  ------   --------------  -----   
 0   carat    1000 non-null   Float32 
 1   cut      1000 non-null   category
 2   color    1000 non-null   category
 3   clarity  1000 non-null   category
 4   depth    1000 non-null   Float32 
 5   table    1000 non-null   Float32 
 6   price    1000 non-null   Int16   
dtypes: Float32(3), Int16(1), category(3)
memory usage: 21.5 KB


In [19]:
dtypes = { # does not include x, y, or z
        "carat": pd.Float32Dtype(),
        "cut": pd.StringDtype(),
        "color": pd.StringDtype(),
        "clarity": pd.StringDtype(),
        "depth": pd.Float32Dtype(),
        "table": pd.Float32Dtype(),
        "price": pd.Int16Dtype(),
}

df_iter = pd.read_csv(
    "../Datasets/diamonds.csv",
    nrows=1_000,
    dtype=dtypes,
    usecols=dtypes.keys(),
    chunksize=200
)

for df in df_iter:
    cat_cols = ["cut", "color", "clarity"]
    df[cat_cols] = df[cat_cols].astype(pd.CategoricalDtype())
    print(f"processed chunk of shape {df.shape}")

processed chunk of shape (200, 7)
processed chunk of shape (200, 7)
processed chunk of shape (200, 7)
processed chunk of shape (200, 7)
processed chunk of shape (200, 7)


In [20]:
def startswith_c(column_name: str) -> bool:
    return column_name.startswith("c")

pd.read_csv(
    "../Datasets/diamonds.csv",
    dtype_backend="numpy_nullable",
    usecols=startswith_c,
)

,carat,cut,color,clarity
0,0.23,Ideal,E,SI2
1,0.21,Premium,E,SI1
2,0.23,Good,E,VS1
3,0.29,Premium,I,VS2
4,0.31,Good,J,SI2
...,...,...,...,...
53935,0.72,Ideal,D,SI1
53936,0.72,Good,D,SI1
53937,0.7,Very Good,D,SI1
53938,0.86,Premium,H,SI2


# **Microsoft Excel – basic reading/writing**

In [21]:
df = pd.DataFrame([
    ["Paul", "McCartney", 1942],
    ["John", "Lennon", 1940],
    ["Richard", "Starkey", 1940],
    ["George", "Harrison", 1943],
], columns=["first", "last", "birth"]).convert_dtypes(dtype_backend="numpy_nullable")

df

,first,last,birth
0,Paul,McCartney,1942
1,John,Lennon,1940
2,Richard,Starkey,1940
3,George,Harrison,1943


In [22]:
df.dtypes

first    string[python]
last     string[python]
birth             Int64
dtype: object

In [23]:
import io

In [25]:
buf = io.BytesIO()
df.to_excel(buf)

In [26]:
buf.seek(0)
pd.read_excel(buf, dtype_backend="numpy_nullable")

,Unnamed: 0,first,last,birth
0,0,Paul,McCartney,1942
1,1,John,Lennon,1940
2,2,Richard,Starkey,1940
3,3,George,Harrison,1943


In [27]:
buf.seek(0)
pd.read_excel(buf, dtype_backend="numpy_nullable", index_col=0)

,first,last,birth
0,Paul,McCartney,1942
1,John,Lennon,1940
2,Richard,Starkey,1940
3,George,Harrison,1943


In [28]:
buf = io.BytesIO()
df.to_excel(buf, index=False)

buf.seek(0)
pd.read_excel(buf, dtype_backend="numpy_nullable")

,first,last,birth
0,Paul,McCartney,1942
1,John,Lennon,1940
2,Richard,Starkey,1940
3,George,Harrison,1943


In [29]:
buf.seek(0)

dtypes = {
    "first": pd.StringDtype(),
    "last": pd.StringDtype(),
    "birth": pd.Int16Dtype(),
}

df = pd.read_excel(buf, dtype=dtypes)

df.dtypes

first    string[python]
last     string[python]
birth             Int16
dtype: object

# **Microsoft Excel – finding tables in non-default locations** 

In [30]:
pd.read_excel(
    "../Datasets/beatles.xlsx",
    dtype_backend="numpy_nullable",
    sheet_name="the_data",
    skiprows=4,
    usecols="C:E",
)

,first,last,birth
0,Paul,McCartney,1942
1,John,Lennon,1940
2,Richard,Starkey,1940
3,George,Harrison,1943


In [32]:
pd.read_excel(
    "../Datasets/beatles.xlsx",
    dtype_backend="numpy_nullable",
    sheet_name=1,
    skiprows=4,
    usecols=["first", "last", "birth"],
)

,first,last,birth
0,Paul,McCartney,1942
1,John,Lennon,1940
2,Richard,Starkey,1940
3,George,Harrison,1943


In [33]:
pd.read_excel(
    "../Datasets/beatles.xlsx",
    dtype_backend="numpy_nullable",
    sheet_name=1,
    skiprows=4,
    usecols="C,D,E",
)

,first,last,birth
0,Paul,McCartney,1942
1,John,Lennon,1940
2,Richard,Starkey,1940
3,George,Harrison,1943


# **Microsoft Excel – hierarchical data**

In [34]:
df = pd.read_excel(
    "../Datasets/hierarchical.xlsx",
    dtype_backend="numpy_nullable",
    index_col=[0, 1],
    header=[0, 1],
)

df

Year                  2024            2025         
Quarter                 Q1      Q2      Q1       Q2
Region  Sub-Region                                 
America East             1       2       4        8
        West            16      32      64      128
        South          256     512    1024     4096
Europe  West          8192   16384   32768    65536
        East        131072  262144  524288  1048576

In [35]:
df.loc[(slice(None), "East"), (slice(None), "Q2")]

,Year,2024,2025
,Quarter,Q2,Q2
Region,Sub-Region,,
America,East,2,8
Europe,East,262144,1048576


# **SQL using SQLAlchemy**

In [36]:
import sqlalchemy as sa

In [37]:
engine = sa.create_engine("sqlite:///:memory:")

In [38]:
df = pd.DataFrame([
    ["dog", 4],
    ["cat", 4],
], columns=["animal", "num_legs"]).convert_dtypes(dtype_backend="numpy_nullable")

df.to_sql(
    "table_name", 
    engine, 
    index=False
)

2

In [39]:
pd.read_sql(
    "table_name",
    engine,
    dtype_backend="numpy_nullable"
)

,animal,num_legs
0,dog,4
1,cat,4


In [40]:
pd.read_sql(
    "SELECT SUM(num_legs) as total_legs from table_name",
    engine,
    dtype_backend="numpy_nullable"
)

,total_legs
0,8


In [42]:
df = pd.DataFrame([
    ["dog", 4],
    ["cat", 4],
    ["human", 2],
], columns=["animal", "num_legs"]).convert_dtypes(dtype_backend="numpy_nullable")

df.to_sql(
    "table_name", 
    engine, 
    index=False, 
    if_exists="replace")

3

In [43]:
new_data = pd.DataFrame([["centipede", 100]], columns=["animal", "num_legs"])

new_data.to_sql(
    "table_name", 
    engine, 
    index=False, 
    if_exists="append")

pd.read_sql(
    "table_name", 
    engine, 
    dtype_backend="numpy_nullable"
)

,animal,num_legs
0,dog,4
1,cat,4
2,human,2
3,centipede,100


# **SQL using ADBC**

In [2]:
from adbc_driver_sqlite import dbapi

In [4]:
df = pd.DataFrame([
    ["dog", 4],
    ["cat", 4],
    ["human", 2],
], columns=["animal", "num_legs"]).convert_dtypes(dtype_backend="numpy_nullable")

df

,animal,num_legs
0,dog,4
1,cat,4
2,human,2


> The term **dbapi** is taken from the Python Database **API Specification** defined in **PEP-249**, which standardizes how Python modules and libraries should be used to interact with databases. Calling the .connect methodwith credentials is the standardized way to open up a database connection in Python.

In [5]:
with dbapi.connect("file::memory:") as conn:
    df.to_sql("table_name", conn, index=False, if_exists="replace")
    df = pd.read_sql(
        "SELECT * FROM table_name",
        conn,
        dtype_backend="numpy_nullable",
    )

df

,animal,num_legs
0,dog,4
1,cat,4
2,human,2


> For smaller datasets, you may not see much of a difference, but the performance gains of **ADBC** will be drastic with larger datasets.

In [6]:
import timeit
import sqlalchemy as sa

In [7]:
np.random.seed(42)
df = pd.DataFrame(
    np.random.randn(10_000, 10),
    columns=list("abcdefghij")
)

df.head()

,a,b,c,d,e,f,g,h,i,j
0,0.496714,-0.138264,0.647689,1.523030,-0.234153,-0.234137,1.579213,0.767435,-0.469474,0.542560
1,-0.463418,-0.465730,0.241962,-1.913280,-1.724918,-0.562288,-1.012831,0.314247,-0.908024,-1.412304
2,1.465649,-0.225776,0.067528,-1.424748,-0.544383,0.110923,-1.150994,0.375698,-0.600639,-0.291694
3,-0.601707,1.852278,-0.013497,-1.057711,0.822545,-1.220844,0.208864,-1.959670,-1.328186,0.196861
4,0.738467,0.171368,-0.115648,-0.301104,-1.478522,-0.719844,-0.460639,1.057122,0.343618,-1.763040


In [8]:
with sa.create_engine("sqlite:///:memory:").connect() as conn:
    func = lambda: df.to_sql("test_table", conn, if_exists="replace")
    print(timeit.timeit(func, number=100))

8.897770573999878


In [9]:
from adbc_driver_sqlite import dbapi

In [10]:
with dbapi.connect("file::memory:") as conn:
    func = lambda: df.to_sql("test_table", conn, if_exists="replace")
    print(timeit.timeit(func, number=100))

1.7556720239999777


> For users wanting to know more about **ADBC**, I recommend viewing my talk from **PyData NYC 2023**, titled Faster SQL with pandas and Apache Arrow, on YouTube (https://youtu.be/XhnfybpWOgA?si=RBrM7UUvpNFyct0L).

# **Apache Parquet**

- As far as a generic storage format for a pd.DataFrame goes, **Apache Parquet** is the best option. Apache Parquet allows:
    - **Metadata storage –** this allows the format to track data types, among other features
    - **Partitioning –** not everything needs to be in one fileQuery support – Parquet files can be queried on disk, so you don’t have to bring all data into memory
    - **Parallelization –** reading data can be parallelized for higher throughput
    - **Compactness –** data is compressed and stored in a highly efficient manner

In [12]:
buf = io.BytesIO()

df = pd.DataFrame([
    ["Paul", "McCartney", 1942],
    ["John", "Lennon", 1940],
    ["Richard", "Starkey", 1940],
    ["George", "Harrison", 1943],
], columns=["first", "last", "birth"]).convert_dtypes(dtype_backend="numpy_nullable")

df

,first,last,birth
0,Paul,McCartney,1942
1,John,Lennon,1940
2,Richard,Starkey,1940
3,George,Harrison,1943


In [13]:
df.to_parquet(buf, index=False)

In [14]:
buf.seek(0)
pd.read_parquet(buf)

,first,last,birth
0,Paul,McCartney,1942
1,John,Lennon,1940
2,Richard,Starkey,1940
3,George,Harrison,1943


> Unlike a format like **CSV**, which only stores data, **the Apache Parquet** format stores both data and **metadata**. Within the metadata, **Apache Parquet** is able to keep track of the data types in use, so whatever data type you write should be exactly what you get back.

In [16]:
df["birth"] = df["birth"].astype(pd.UInt16Dtype())

df.dtypes

first    string[python]
last     string[python]
birth            UInt16
dtype: object

In [17]:
buf = io.BytesIO()
df.to_parquet(buf, index=False)

buf.seek(0)
pd.read_parquet(buf).dtypes

first    string[python]
last     string[python]
birth            UInt16
dtype: object

In [18]:
suboptimal_df = pd.DataFrame([
    [0, "foo"],
    [1, "bar"],
    [2, "baz"],
], columns=["int_col", "str_col"])

buf = io.BytesIO()
suboptimal_df.to_parquet(buf, index=False)

buf.seek(0)
pd.read_parquet(buf, dtype_backend="numpy_nullable").dtypes

int_col             Int64
str_col    string[python]
dtype: object

In [19]:
pd.read_parquet("../Datasets/partitions/2022/q1_sales.parquet")

,year,quarter,region,sales
0,2022,Q1,America,1
1,2022,Q1,Europe,2


In [20]:
pd.read_parquet('../Datasets/partitions/')

,year,quarter,region,sales
0,2022,Q1,America,1
1,2022,Q1,Europe,2
2,2022,Q2,America,4
3,2022,Q2,Europe,8
4,2023,Q1,America,16
5,2023,Q1,Europe,32
6,2023,Q2,America,64
7,2023,Q2,Europe,128


In [21]:
pd.read_parquet(
    "../Datasets/partitions/",
    filters=[("region", "==", "Europe")],
)

,year,quarter,region,sales
0,2022,Q1,Europe,2
1,2022,Q2,Europe,8
2,2023,Q1,Europe,32
3,2023,Q2,Europe,128


# **JSON**

In [22]:
import json

In [23]:
beatles = {
    "first": ["Paul", "John", "Richard", "George",],
    "last": ["McCartney", "Lennon", "Starkey", "Harrison",],
    "birth": [1942, 1940, 1940, 1943],
}

serialized = json.dumps(beatles)
print(f"serialized values are: {serialized}")

deserialized = json.loads(serialized)
print(f"deserialized values are: {deserialized}")

serialized values are: {"first": ["Paul", "John", "Richard", "George"], "last": ["McCartney", "Lennon", "Starkey", "Harrison"], "birth": [1942, 1940, 1940, 1943]}
deserialized values are: {'first': ['Paul', 'John', 'Richard', 'George'], 'last': ['McCartney', 'Lennon', 'Starkey', 'Harrison'], 'birth': [1942, 1940, 1940, 1943]}


In [24]:
data = io.StringIO(serialized)
pd.read_json(data, dtype_backend="numpy_nullable")

,first,last,birth
0,Paul,McCartney,1942
1,John,Lennon,1940
2,Richard,Starkey,1940
3,George,Harrison,1943


In [25]:
df = pd.DataFrame(beatles)
print(df.to_json())

{"first":{"0":"Paul","1":"John","2":"Richard","3":"George"},"last":{"0":"McCartney","1":"Lennon","2":"Starkey","3":"Harrison"},"birth":{"0":1942,"1":1940,"2":1940,"3":1943}}


- For these use cases and more, pandas allows you to pass an argument to **orient=** , whose value dictates the layout of the JSON to be read or written:
    - **columns (default):** Produces JSON objects, where the key is a column label and the value is another object that maps the row label to a data point.
    - **records :** Each row of the pd.DataFrame is represented as a JSON array, containing objects that map column names to a data point.
    - **split :** Maps to {"columns": [...], "index": [...], "data": [...]} . Columns/index values are arrays of labels, and data contains arrays of arrays.
    - **index :** Similar to columns, except that the usage of row and column labels as keys is reversed.
    - **values :** Maps the data of a pd.DataFrame to an array of arrays. Row/column labels are dropped.
    - **table :** Adheres to the JSON Table Schema.

In [26]:
df = pd.DataFrame(
    beatles, 
    index=["row 0", "row 1", "row 2", "row 3"]
).convert_dtypes(dtype_backend="numpy_nullable")

df

,first,last,birth
row 0,Paul,McCartney,1942
row 1,John,Lennon,1940
row 2,Richard,Starkey,1940
row 3,George,Harrison,1943


In [27]:
serialized = df.to_json(orient="columns")

print(f'Length of orient="columns": {len(serialized)}')

serialized[:100]

Length of orient="columns": 221


'{"first":{"row 0":"Paul","row 1":"John","row 2":"Richard","row 3":"George"},"last":{"row 0":"McCartn'

In [28]:
pd.read_json(
    io.StringIO(serialized),
    orient="columns",
    dtype_backend="numpy_nullable"
)

,first,last,birth
row 0,Paul,McCartney,1942
row 1,John,Lennon,1940
row 2,Richard,Starkey,1940
row 3,George,Harrison,1943


In [29]:
serialized = df.to_json(orient="records")

print(f'Length of orient="records": {len(serialized)}')

serialized[:100]

Length of orient="records": 196


'[{"first":"Paul","last":"McCartney","birth":1942},{"first":"John","last":"Lennon","birth":1940},{"fi'

In [30]:
pd.read_json(
    io.StringIO(serialized),
    orient="orient",
    dtype_backend="numpy_nullable"
)

,first,last,birth
0,Paul,McCartney,1942
1,John,Lennon,1940
2,Richard,Starkey,1940
3,George,Harrison,1943


In [31]:
serialized = df.to_json(orient="split")

print(f'Length of orient="split": {len(serialized)}')

serialized[:100]

Length of orient="split": 190


'{"columns":["first","last","birth"],"index":["row 0","row 1","row 2","row 3"],"data":[["Paul","McCar'

In [32]:
pd.read_json(
    io.StringIO(serialized),
    orient="split",
    dtype_backend="numpy_nullable",
)

,first,last,birth
row 0,Paul,McCartney,1942
row 1,John,Lennon,1940
row 2,Richard,Starkey,1940
row 3,George,Harrison,1943


In [33]:
serialized = df.to_json(orient="index")

print(f'Length of orient="index": {len(serialized)}')

serialized[:100]

Length of orient="index": 228


'{"row 0":{"first":"Paul","last":"McCartney","birth":1942},"row 1":{"first":"John","last":"Lennon","b'

In [34]:
pd.read_json(
    io.StringIO(serialized),
    orient="index",
    dtype_backend="numpy_nullable",
)

,first,last,birth
row 0,Paul,McCartney,1942
row 1,John,Lennon,1940
row 2,Richard,Starkey,1940
row 3,George,Harrison,1943


> Generally, **orient="index"** will take up more space than **orient="columns"** , since most **pd.DataFrame** objects use column labels that are more verbose than index labels. I would only advise using this format in the possibly rare instances where your column labels are less verbose, or if you have strict formatting requirements imposed by another system.

In [35]:
serialized = df.to_json(orient="values")

print(f'Length of orient="values": {len(serialized)}')

serialized[:100]

Length of orient="values": 104


'[["Paul","McCartney",1942],["John","Lennon",1940],["Richard","Starkey",1940],["George","Harrison",19'

In [36]:
pd.read_json(
    io.StringIO(serialized),
    orient="values",
    dtype_backend="numpy_nullable",
)

,0,1,2
0,Paul,McCartney,1942
1,John,Lennon,1940
2,Richard,Starkey,1940
3,George,Harrison,1943


In [37]:
serialized = df.to_json(orient="table")

print(f'Length of orient="table": {len(serialized)}')

serialized[:100]

Length of orient="table": 530


'{"schema":{"fields":[{"name":"index","type":"string"},{"name":"first","type":"string","extDtype":"st'

> The Table Schema is more verbose because it stores metadata about the data being serialized, similar to what we saw with the Apache Parquet format (although with fewer features than **Apache Parquet**). With all of the other **orient=** arguments, pandas would have to infer the type of data as it is being read, but the **JSON Table Format** preserves that information for you. As such, you don’t even need the **dtype_backend="numpy_nullable"** argument, assuming you used the pandas extension types to begin with

In [38]:
df["birth"] = df["birth"].astype(pd.UInt16Dtype())
serialized = df.to_json(orient="table")

pd.read_json(
    io.StringIO(serialized),
    orient="table",
).dtypes

first    string[python]
last     string[python]
birth            UInt16
dtype: object

In [39]:
data = {
    "records": [{
        "name": "human",
        "characteristics": {
            "num_leg": 2,
            "num_eyes": 2
        }
    }, {
        "name": "dog",
        "characteristics": {
            "num_leg": 4,
            "num_eyes": 2
        }
    }, {
        "name": "horseshoe crab",
        "characteristics": {
            "num_leg": 10,
            "num_eyes": 10
        }
    }],
    "type": "animal",
    "pagination": {
        "next": "23978sdlkusdf97234u2io",
        "has_more": 1
    }
}

In [40]:
pd.json_normalize(
    data,
    record_path="records"
).convert_dtypes(dtype_backend="numpy_nullable")

,name,characteristics.num_leg,characteristics.num_eyes
0,human,2,2
1,dog,4,2
2,horseshoe crab,10,10


In [41]:
pd.json_normalize(
    data,
    record_path="records",
    meta="type"
).convert_dtypes(dtype_backend="numpy_nullable")

,name,characteristics.num_leg,characteristics.num_eyes,type
0,human,2,2,animal
1,dog,4,2,animal
2,horseshoe crab,10,10,animal


# **HTML**

> Wikipedia blocks requests that use default or missing User-Agent headers (which Pandas uses via standard urllib), resulting in an HTTP Error 403: Forbidden.
Pass a custom User-Agent header via the storage_options parameter, or fetch the HTML using requests first.

### **First Option**

In [3]:
url = "https://en.wikipedia.org/wiki/The_Beatles#Discography"
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}

dfs = pd.read_html(
    url, 
    storage_options={"User-Agent": headers["User-Agent"]}, 
    dtype_backend="numpy_nullable"
)

print(len(dfs))

72


### **Second Option**

In [4]:
import requests

In [6]:
url = "https://en.wikipedia.org/wiki/The_Beatles#Discography"
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}

response = requests.get(url, headers=headers)

# Wrap response.text in StringIO
dfs = pd.read_html(io.StringIO(response.text), dtype_backend="numpy_nullable")

print(len(dfs))

72


> Contrary to the other I/O methods we have seen so far, **pd.read_html** doesn’t return a pd.DataFrame but, instead, returns a list of pd.DataFrame objects.

In [7]:
dfs[0]

,The Beatles,The Beatles.1
0,"The Beatles in 1963 From left: Paul McCartney,...","The Beatles in 1963 From left: Paul McCartney,..."
1,Background information,Background information
2,Origin,"Liverpool, England"
3,Genres,Rockpopbeatpsychedelia
4,Works,Albumssinglessongscoversfilmrecording sessions...
5,Years active,1960–1970
6,Labels,PolydorParlophoneTollieVee-JayCapitolSwanOdeon...
7,Spinoff of,The Quarrymen
8,Awards,Full list
9,<NA>,<NA>


In [10]:
# Option 1: Match by text content unique to this table (e.g., 'Associated places' or 'Fifth Beatle')
url = "https://en.wikipedia.org/wiki/The_Beatles#Discography"
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}

dfs = pd.read_html(
    url, 
    storage_options={"User-Agent": headers["User-Agent"]},
    attrs={"class": "navbox-inner"},
    dtype_backend="numpy_nullable"
)

dfs[0]

,vteThe Beatles,vteThe Beatles.1
0,John Lennon Paul McCartney George Harrison Rin...,John Lennon Paul McCartney George Harrison Rin...
1,History,The Quarrymen In Hamburg At the Cavern Club De...
2,Lists,Awards and nominations Bootlegs Albums Singles...
3,Tours and performances,1960 Johnny Gentle Tour Winter 1963 Helen Shap...
4,Personnel,Management Neil Aspinall Tony Barrow Peter Ben...
5,Management,Neil Aspinall Tony Barrow Peter Bennett Peter ...
6,Production,Geoff Emerick Bert Kaempfert Richard Lush Geor...
7,Associated companies,Apple Corps Apple Records Capitol Records EMI ...
8,Associated places,10 Admiral Grove 12 Arnold Grove 20 Forthlin R...
9,Selected books,The Beatles Anthology The Beatles: The Authori...


In [11]:
dfs[0].filter(regex=r"Title|UK|AUS|CAN").head()

""
0
1
2
3
4


In [12]:
url = "https://en.wikipedia.org/wiki/The_Beatles#Discography"
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}

dfs = pd.read_html(
    url, 
    storage_options={"User-Agent": headers["User-Agent"]},
    attrs={"class": "navbox-inner"},
    header=1,
    dtype_backend="numpy_nullable"
)

dfs[0]

,John Lennon Paul McCartney George Harrison Ringo Starr Stuart Sutcliffe Pete Best Chas Newby Norman Chapman Tommy Moore Jimmie Nicol Outline Timeline,John Lennon Paul McCartney George Harrison Ringo Starr Stuart Sutcliffe Pete Best Chas Newby Norman Chapman Tommy Moore Jimmie Nicol Outline Timeline.1
0,History,The Quarrymen In Hamburg At the Cavern Club De...
1,Lists,Awards and nominations Bootlegs Albums Singles...
2,Tours and performances,1960 Johnny Gentle Tour Winter 1963 Helen Shap...
3,Personnel,Management Neil Aspinall Tony Barrow Peter Ben...
4,Management,Neil Aspinall Tony Barrow Peter Bennett Peter ...
5,Production,Geoff Emerick Bert Kaempfert Richard Lush Geor...
6,Associated companies,Apple Corps Apple Records Capitol Records EMI ...
7,Associated places,10 Admiral Grove 12 Arnold Grove 20 Forthlin R...
8,Selected books,The Beatles Anthology The Beatles: The Authori...
9,Other topics,Apple Corps v Apple Computer Apple scruffs Bea...


# **Pickle**